In [ ]:
from google.colab import files

# 📌 Opens a file upload dialog in Google Colab
uploaded = files.upload()  # A window will open to upload files

Saving mixtec-train.txt to mixtec-train.txt
Saving mixtec-val.txt to mixtec-val.txt
Saving spanish-train.txt to spanish-train.txt
Saving spanish-val.txt to spanish-val.txt


In [ ]:
# 📌 Install necessary libraries
!pip install datasets transformers huggingface_hub --upgrade

import pandas as pd
import json
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer, TrainingArguments, Trainer, AutoModelForSeq2SeqLM
import wandb
import torch

# 📌 Read training files
with open("mixtec-train.txt", "r", encoding="utf-8") as f_mix_train, open("spanish-train.txt", "r", encoding="utf-8") as f_esp_train:
    mixtec_train = f_mix_train.readlines()
    spanish_train = f_esp_train.readlines()

# 📌 Ensure both training files have the same number of lines
assert len(mixtec_train) == len(spanish_train), "⚠️ Error: Training files have a different number of lines."

# 📌 Read validation files
with open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val, open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val:
    mixtec_val = f_mix_val.readlines()
    spanish_val = f_esp_val.readlines()

# 📌 Ensure both validation files have the same number of lines
assert len(mixtec_val) == len(spanish_val), "⚠️ Error: Validation files have a different number of lines."

# 📌 Create DataFrames for training and validation
train_df = pd.DataFrame({"spanish": [s.strip() for s in spanish_train], "mixtec": [m.strip() for m in mixtec_train]})
val_df = pd.DataFrame({"spanish": [s.strip() for s in spanish_val], "mixtec": [m.strip() for m in mixtec_val]})

# 📌 Convert DataFrames into Hugging Face dataset format
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 📌 Create a DatasetDict structure
dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

# 📌 Load the mBART-50 tokenizer
model_name = "facebook/mbart-large-50"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 📌 Set source and target languages
tokenizer.src_lang = "es_XX"  # 📌 Input language: Spanish
tokenizer.tgt_lang = "mix_Latn"  # 📌 Target language: Mixtec

# 📌 Preprocessing function
def preprocess_function(examples):
    inputs = examples["spanish"]  # 📌 Input text in Spanish
    targets = examples["mixtec"]  # 📌 Target text in Mixtec

    # Tokenize input and target texts
    model_inputs = tokenizer(inputs, padding="max_length", truncation=True, max_length=128)
    labels = tokenizer(targets, padding="max_length", truncation=True, max_length=128)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# 📌 Tokenize dataset
tokenized_datasets = dataset.map(preprocess_function, batched=True)

# 📌 Load mBART-50 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 📌 Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🔍 Using device: {device}")
model.to(device)

# 📌 Training configuration
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    per_device_train_batch_size=8,  # 📌 Adjust batch size based on available memory
    per_device_eval_batch_size=8,
    num_train_epochs=5,  # 📌 Increase epochs if dataset is small
    logging_dir="./logs",
    logging_steps=50,  # 📌 Reduce log frequency for better stability
    learning_rate=3e-5,  # 📌 Fine-tuning learning rate
    weight_decay=0.01,  # 📌 Regularization to prevent overfitting
    warmup_ratio=0.06,  # 📌 Warmup for 6% of training steps
    fp16=True,  # 📌 Enable FP16 for faster training if using GPU
    report_to="wandb",  # 📌 Log metrics to Weights & Biases
    push_to_hub=False
)

# 📌 Initialize Weights & Biases
wandb.login()
wandb.init(project="mbart_finetuning_espanol_mixteco", name="mbart_train_run")

# 📌 Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    tokenizer=tokenizer,
)

# 📌 Start training
trainer.train()

# 📌 Save fine-tuned model
model.save_pretrained("./mbart_modelo_espanol_mixteco")
tokenizer.save_pretrained("./mbart_modelo_espanol_mixteco")

print("✅ Training completed and model saved.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.0/468.0 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 17.8 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface-hub 0.28.1
    Uninstalling huggingface-hub-0.28.1:
      Successfully uninstalled huggingface-hub-0.28.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.48.3
    Uninstalling transformers-4.48.3:
      Successfully uninstalled transformers-4.48.3


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

...

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: hsantiago13 (hsantiago13-university-aut-noma-de-quer-taro) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


<ipython-input-2-4c1066ec6c04>:92: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Epoch,Training Loss,Validation Loss
1,0.822000,0.238808
2,0.627700,0.216874
3,0.555300,0.211243
4,0.466200,0.214233
5,0.405300,0.221758


/usr/local/lib/python3.11/dist-packages/transformers/modeling_utils.py:2810: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 200, 'early_stopping': True, 'num_beams': 5}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(


✅ Entrenamiento completado y modelo guardado.


In [ ]:
# 📌 Install necessary packages
!pip install datasets transformers huggingface_hub sacrebleu sacremoses evaluate --upgrade

# Importing the necessary libraries for data handling, model loading, and evaluation
import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import sacrebleu
import evaluate

# 📌 Load pre-trained model and tokenizer
model_name = "./mbart_modelo_espanol_mixteco"  # Path to the fine-tuned model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# 📌 Check for GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 📌 Read validation data
with open("spanish-val.txt", "r", encoding="utf-8") as f_esp_val, open("mixtec-val.txt", "r", encoding="utf-8") as f_mix_val:
    spanish_val = [line.strip() for line in f_esp_val.readlines()]
    mixtec_val = [line.strip() for line in f_mix_val.readlines()]

# 📌 Generate translations with the model
def translate_texts(texts, model, tokenizer, device):
    translations = []
    for text in texts:
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=128)
        inputs = {key: value.to(device) for key, value in inputs.items()}  # Send to GPU if available

        with torch.no_grad():
            output = model.generate(**inputs)  # 🔹 Removed forced_bos_token_id

        translated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        translations.append(translated_text)
    return translations

# 📌 Get the generated translations
generated_translations = translate_texts(spanish_val, model, tokenizer, device)

# 📌 Evaluation using BLEU
bleu_score = sacrebleu.corpus_bleu(generated_translations, [mixtec_val])
print(f"🔹 BLEU Score: {bleu_score.score:.2f}")

# 📌 Evaluation using TER (Translation Edit Rate)
ter_metric = evaluate.load("ter")  # ✅ Changed from load_metric to evaluate.load
ter_score = ter_metric.compute(predictions=generated_translations, references=mixtec_val)
print(f"🔹 TER Score: {ter_score['score']:.2f}")

🔹 BLEU Score: 4.20


🔹 TER Score: 98.99
